In [ ]:
import scipy
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

from sklearn import linear_model    # Herramientas de modelos lineales
from sklearn.metrics import mean_squared_error, r2_score    # Medidas de desempeño
from sklearn.preprocessing import StandardScaler    # Herramientas de polinomios

from sklearn.model_selection import train_test_split, KFold, cross_val_score

from formulaic import Formula

In [ ]:
# Si necesitan instalar algún paquete
#!pip install gapminder
#!pip install formulaic

In [ ]:
# Si necesitan cambiar de directorio de trabajo
#import os
#print(pwd)
#os.chdir('./notebooks')

# Caso de estudio: calorías de alimentos

In [ ]:
df_nutricion = pd.read_csv('nutrition.csv')
df_nutricion

En este ejemplo consideramos que los datos faltantes representan que el alimento no contiene ese ingrediente y lo convertimos a 0.

In [ ]:
# Utilizamos fillna para convertir NaN a 0.
df_nutricion = df_nutricion.fillna(0)

In [ ]:
df_nutricion.head()

Construimos las matrices X e y utilizando Formulaic

In [ ]:
# No usamos intercept en este modelo
y, X = (
    Formula('Calorias_kcal ~ Proteinas_g + Carbohidratos_g + GrasaTotal_g + Colesterol_mg + Fibra_g + Agua_g + Alcohol_g + VitaminaC_mg - 1')
    .get_model_matrix(df_nutricion)
)

In [ ]:
X.head()

In [ ]:
y.head()

Antes de separar en entrenamiento y testeo, veamos los errores del modelo lineal con todos los datos 

In [ ]:
modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal. 
modelo.fit(X, y)   # Realizamos el ajuste

Analizamos la "bondad" del ajuste.

In [ ]:
y_pred = modelo.predict(X)

# Calculando el R^2
r2 = r2_score(y, y_pred)
print('R^2: ', r2)

# Calculando la RECM
ecm = mean_squared_error(y, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

A priori es un buen modelo, tenemos 7792 observaciones y obtenemos R^2 casi igual a 1 con solo 9 variables.

In [ ]:
modelo.coef_

Vemos que las variables Proteinas_g, Carbohidratos_g, GrasaTotal_g y Alcohol_g son las que tienen coeficientes mas grandes

# Conjuntos de entrenamiento y testeo
Ajustamos el modelo separando en 80-20

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# Construimos primero las matrices X e y utilizando Formulaic (o cualquier otro método)
y, X = (
    Formula('Calorias_kcal ~ Proteinas_g + Carbohidratos_g + GrasaTotal_g + Colesterol_mg + Fibra_g + Agua_g + Alcohol_g + VitaminaC_mg - 1')
    .get_model_matrix(df_nutricion)
)

# Separamos en entrenamiento (train) y testeo (test).
# El parámetro test_size=0.2 indica que tomamos un 20% de los datos para testeo.
# El parámetro random_state=42 es lo que se denomina semilla aleatoria. 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

In [ ]:
X_train

In [ ]:
X_test

### Números pseudo-aleatorios y semillas aleatorias.

Las computadoras no pueden generar números al azar, tienen algoritmos que generan números que parecen al azar denominados pseudo aleatorios. 

Los números se generan a partir de una semilla. Si corremos el codigo utilizando la misma semilla,  vamos a obtener siempre el mismo resultado.

Esto permite que el experimento sea reproducible.

### Entrenamiento

In [ ]:
# Entrenamos el modelo utilizando los conjuntos de entrenamiento

modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal
modelo.fit(???)   # Realizamos el ajuste

### Testeo

In [ ]:
# Medimos la bondad del ajuste en el conjunto de testeo

y_pred = modelo.predict(???)

# Calculando el R^2
r2 = r2_score(???)
print('R^2: ', r2)

# Calculando el ECM
ecm = mean_squared_error(???)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

Vemos que el modelo ajusto bien en los datos de testeo, podemos confiar en el modelo obtenido.

**Ejercicio:** Utilizar distintas semillas aleatorias. ¿Se obtienen los mismos valores? ¿Se mantienen las conclusiones?

## Selección de modelos

Comparamos con un modelo utilizando solo las cuatro variables con mayores coeficientes y sin intercept.

In [ ]:
y, X = (
    Formula('Calorias_kcal ~ Proteinas_g + Carbohidratos_g + GrasaTotal_g + Alcohol_g - 1')
    .get_model_matrix(df_nutricion)
)
X.head()

In [ ]:
# Separamos en entrenamiento (train) y testeo (test).
# El parámetro test_size=0.2 indica que tomamos un 20% de los datos para testeo.
# El parámetro random_state=42 es lo que se denomina semilla aleatoria. 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo.fit(???)   # Realiza
print("Coeficientes:", modelo.coef_)

y_pred = modelo.predict(X_test)
# Calculando el R^2
r2 = r2_score(y_test, y_pred)
print('R^2: ', r2)

# Calculando el ECM
ecm = mean_squared_error(y_test, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

Vemos que el modelo es un poco peor pero mucho más simple. 

**Conclusión rápida:** El modelo con 3 variables es útil para una cuenta rápida, pero si necesitamos una mayor precisión podemos utilizar el modelo completo.

**Ejercicio.** Buscar en recursos en-línea la fórmula usualmente utilizada para el cálculo de calorías y comparar con la fórmula que obtuvimos.

## Ejercicio

En el dataset de diamantes, vimos que las variables x, y y z están muy correlacionadas. Mejorará el modelo si eliminamos alguna de ellas? (variables correlacionadas pueden tener coeficientes grandes en el modelo con signos opuestos y eso aumenta los errores)

In [ ]:
df_diamonds = sns.load_dataset('diamonds')
df_diamonds

In [ ]:
# Modelo con todas las variables numéricas
y, X = (
    Formula("price ~ carat + depth + table + x + y + z - 1")
    .get_model_matrix(df_diamonds)
)
y = y.squeeze()  # Para convertir a serie

X.head()

In [ ]:
# Separamos en entrenamiento y testeo


In [ ]:
# Ajustamos el modelo en el conjunto de entrenamiento y calculamos los errores en el conjunto de testeo


In [ ]:
# Repetimos para el modelo sin alguna de las variables x, y o z


In [ ]:
# Y mejora el modelo si agregamos alguna de las variables categóricas?


# Caso de estudio: rendimiento del suelo

En una región agrícola queremos predecir el rendimiento del suelo en función del uso de distintos fertilizantes y otras variables.
Se hacen distintas mediciones y se juntan los datos en una tabla.

¿Qué variables usarían si quieren un modelo con buena capacidad predictiva y pocas variables?

Proponer un modelo para predecir el rendimiento de una hectárea cultivada en función de algunas características del lugar y los fertilizantes utilizados.

**Nota:** La variable Espigas es también una medida de rendimiento, no puede usarse como variable independiente.

In [ ]:
df_rendimiento = pd.read_csv('rendimiento.csv')
df_rendimiento

In [ ]:
df_rendimiento.info()

In [ ]:
# Acá están todas las variables
"Rendimiento ~ pH + MO + P + N + S + P_Ferti + P_kgha + S_Fuente + S_kgha + Ndisp + Palc + Variedad + FdS + plantas + momento + IncSep + IncMam + IncRoya"

# Cuatro niveles de selección de modelos
Queremos estimar los costos de salud que tendrá un cliente de una prepaga en función de algunas variables de la persona.

In [ ]:
df_salud = pd.read_csv('insurance.csv')
df_salud

La variable respuesta es `charges` y el resto son variables explicativas.

**Pregunta:** ¿Cuáles son variables numércias? ¿Cuáles son categóricas? Las variables categóricas son: ¿binarias, nominales, ordinales?

In [ ]:
df_salud.info()

**Pregunta:** ¿Cuántas categorías tiene la variable `region`?

In [ ]:
# Otro comando útil para analizar las variables numéricas
df_salud.describe()

Por ejemplo, en la variable `charges` vemos que el promedio es 13270, y el máximo es 63770. Esto podría indica la presencia de outliers, que dificultan el modelo. ¿Cómo podemos visualizar los outliers?

En este ejemplo dejamos los outliers. Ejercicio: ¿cambian las conclusiones si eliminamos los outliers?

## Nivel 1: entrenamos y testeamos el modelo usando todos los datos

In [ ]:
y, X = (
    Formula('charges ~ age + sex + bmi + children + smoker + region')
    .get_model_matrix(df_salud)
)
X.head()

In [ ]:
# Podemos ver la correlación entre las distintas variables (corresponde al R de un modelo lineal y=ax+b)
pd.concat([X,y], axis = 1).corr()

In [ ]:
# Ajustamos el modelo
modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo.fit(X, y)   # Realiza
print("Coeficientes:", modelo.coef_)

In [ ]:
# Predicciones
y_pred = modelo.predict(X)

# Bondad del ajuste
r2 = r2_score(y, y_pred)
print('R^2: ', r2)
ecm = mean_squared_error(y, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

## Nivel 2: separamos en entrenamiento y testeo

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Ajustamos el modelo
modelo = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo.fit(X_train, y_train)   # Realiza

# Predicciones
y_pred = modelo.predict(X_test)

# Bondad del ajuste
r2 = r2_score(y_test, y_pred)
print('R^2: ', r2)
ecm = mean_squared_error(y_test, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

**Pregunta:** Con la semilla aleatoria 42 bajó el ECM. Si cambiamos la semilla por ejemplo a 4?

## Nivel 3: separamos en entrenamiento, validación y testeo

Para crear los conjuntos podemos hacer aplicando estos pasos:
1. Separamos todo el conjunto de datos (dataframe completo) en entrenamiento y testeo.
2. A partir del dataframe de entrenamiento/validación, construimos las matrices y, X del modelo (o y1, X1 si luego vamos a comparar con otro modelo y2, X2)
3. Separamos las matrices y, X en entrenamiento y validacion (y_train, X_train, y_val, X_val)
4. Ajustamos el modelo con y_train, X_train y vemos las predicciones en y_test, X_test
5. Probamos varios modelos y elegimos el que tiene menor pérdida en validación
6. A partir del dataframe de testeo, construimos las matrices y_test, X_test con las columnas que obtuvimos en el modelo ganandor.
7. Ajustamos la fórmula en todo los datos de entrenamiento (matrices y, X) y calculamos las predicciones en testeo y las medidas de bondad del modelo elegido.

#### Paso 1: separamos en entrenamiento y testeo el dataframe original

In [ ]:
df_train, df_test = train_test_split(df_salud, test_size=0.2, random_state=42)
df_train.shape

Si realizamos alguna normalización de las variables, tenemos que tener cuidado y evitar usar en el entrenamiento datos del conjunto de testeo.

Por ejemplo, si queremos normalizar una variable mediante StandardScaler, tenemos que calcular los parámetros de la normalización en los datos de entrenamiento y aplicar esa misma transformación en los datos de testeo.

Como hacemos modelo lineal, no es importante en este caso normalizar variables (el modelo lineal es invariante por transformaciones lineales).

#### Paso 2A: definimos un primer modelo y separamos el dataset df_train en entrenamiento y validación para entrenar el modelo.

In [ ]:
formula1 = 'charges ~ age + sex + bmi + children + smoker + region'
y1, X1 = (
    Formula(formula1)
    .get_model_matrix(df_train)
)

In [ ]:
X_train_1, X_val_1, y_train_1, y_val_1 = train_test_split(X1, y1, test_size=0.2, random_state=42)
X_train.shape

In [ ]:
modelo1 = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo1.fit(???)   # Realizamos el ajuste

# Predicciones
y_pred = modelo1.predict(???)

# Bondad del ajuste
r2 = r2_score(???)
print('R^2: ', r2)
ecm = mean_squared_error(???)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

#### Paso 2B: definimos otro modelo y repetimos el paso 2A

In [ ]:
# Las variables sexo y región no relacionadas a los gastos
formula2 = 'charges ~ age + bmi + children + smoker'
y2, X2 = (
    Formula(formula2)
    .get_model_matrix(df_train)
)
X_train_2, X_val_2, y_train_2, y_val_2 = train_test_split(X2, y2, test_size=0.2, random_state=42)
X_train.shape

In [ ]:
modelo2 = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
modelo2.fit(X_train_2, y_train_2)   # Realizamos el ajuste

# Predicciones
y_pred = modelo2.predict(X_val_2)

# Bondad del ajuste
r2 = r2_score(y_val_2, y_pred)
print('R^2: ', r2)
ecm = mean_squared_error(y_val_2, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

De los modelos probados, nos quedamos con el de menor RECM.

#### Paso 3: analizamos como funciona el modelo en el conjunto de testeo.

**Importante:** Para esto, entrenamos el modelo ganador utilizando **TODOS** los datos de entrenamiento (el modelo es la fórmula, no los coeficientes).

**Recordar:** mientras mas datos usamos para entrenar, mejor!

In [ ]:
# Ajustamos nuestro modelo ganador en TODO el conjunto de entrenamiento. 
modelo1.fit(???)

# Realizamos las mismas transformaciones en el conjunto de testeo
y_test, X_test = (
    Formula(formula???)
    .get_model_matrix(???)
)

# Predicciones
y_pred = modelo1.predict(???)

# Bondad del ajuste
r2 = r2_score(y_test, y_pred)
print('R^2: ', r2)
ecm = mean_squared_error(y_test, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

## Nivel 4: separamos en entrenamiento y testeo, y hacemos validación cruzada en el conjunto de entrenamiento.
### Paso 1: separamos en entrenamiento y testeo el dataframe original

In [ ]:
# La misma separación del Nivel 3
df_train, df_test = train_test_split(df_salud, test_size=0.2, random_state=42)
df_train.shape

#### Paso 2A: definimos un primer modelo y lo ajustamos por validación cruzada en el conjunto de entrenamiento.

In [ ]:
formula1 = 'charges ~ age + sex + bmi + children + smoker + region'
y1, X1 = (
    Formula(formula1)
    .get_model_matrix(df_train)
)

In [ ]:
# Definimos los subconjuntos para la validación cruzada.
# Utilizamos KFold de sklearn
cv = KFold(n_splits=5, random_state=42, shuffle=True)

## Generadores e iteradores perezosos en Python
Nos detenemos un momento para entender qué nos devuelve KFold

In [ ]:
# Esto solo nos muestra las opciones que utilizamos
cv

In [ ]:
# La forma de utilizado es a través del método split
pliegos = cv.split(X1)
pliegos

`split` nos devuelve un "generador", esto es un **iterador perezoso** (lazy iterator).

Los iteradores perezosos son objetos que se pueden recorrer como una lista. 

Sin embargo, a diferencia de las listas, los iteradores perezosos no almacenan su contenido en la memoria, lo van generando a medida que lo necesitamos.


In [ ]:
# Podemos acceder a los elementos a través de la función next
next(pliegos)

In [ ]:
# Pero lo mas común es utilizarlos en un ciclo:
pliegos = cv.split(X1)
for train_index, test_index in pliegos:
    print(test_index[0:10])

In [ ]:
# Ahora no quedo nada, ya generó todo lo que tenía para generar
next(pliegos)

In [ ]:
# Acá tampoco hay nada... pero por qué no da error?
# Cómo hago para que funcione?
for train_index, test_index in pliegos:
    print(test_index[0:10])

#### Volvemos al Paso 2A

In [ ]:
# Para seleccionar algunas filas dados los índices, utilizamos iloc
for train_index, val_index in cv.split(X1):
    X_train, X_val, y_train, y_val = X1.iloc[train_index], X1.iloc[val_index], y1.iloc[train_index], y1.iloc[val_index]
    
    # Acá tenemos que hacer el ajuste y la predicción para cada pliego
    # ...

Agregamos el codigo para ajuste y predicción

In [ ]:
modelo1 = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
rmse1 = np.zeros(cv.get_n_splits())  # Vamos a guardar el error en cada pliego

ind = 0

# Para seleccionar algunas filas dados los índices, utilizamos iloc
for train_index, val_index in cv.split(X1):
    X_train, X_val, y_train, y_val = X1.iloc[train_index], X1.iloc[val_index], y1.iloc[train_index], y1.iloc[val_index]
    modelo1.fit(X_train, y_train)
    
    y_pred = modelo1.predict(X_val)
    rmse1[ind] = np.sqrt(mean_squared_error(y_val, y_pred))  # Guardamos los errores en un vector
    ind = ind + 1

In [ ]:
print(rmse1)

In [ ]:
print(rmse1.mean())  # Este es el valor que queremos minimizar

#### Paso 2B: definimos otro modelo y repetimos el paso 2A

In [ ]:
formula2 = 'charges ~ age + bmi + children + region + smoker'
y2, X2 = (
    Formula(formula2)
    .get_model_matrix(df_train)
)

cv = KFold(n_splits=5, random_state=42, shuffle=True)  # No es necesario definirlo nuevamente, solo para recordar que era.

modelo2 = linear_model.LinearRegression(fit_intercept = False)    # Inicializamos un modelo de Regresion Lineal sin intercept
rmse2 = np.zeros(cv.get_n_splits())  # Vamos a guardar el error en cada pliego

ind = 0

# Para seleccionar algunas filas dados los índices, utilizamos iloc (lo vimos en la clase 2)
for train_index, test_index in cv.split(X2):
    X_train, X_val, y_train, y_val = X2.iloc[train_index], X2.iloc[val_index], y2.iloc[train_index], y2.iloc[val_index]
    modelo2.fit(X_train, y_train)
    
    y_pred = modelo2.predict(X_val)
    rmse2[ind] = np.sqrt(mean_squared_error(y_val, y_pred))
    ind = ind + 1

In [ ]:
print(rmse2)
print(rmse2.mean())  # Este es el valor que queremos minimizar

De los modelos probados, nos quedamos con el de menor RECM. 

#### Paso 3: analizamos como funciona el modelo en el conjunto de testeo.

Copiamos el mismo código del paso 3 del nivel 3.

In [ ]:
# Ajustamos nuestro modelo ganador en TODO el conjunto de entrenamiento. 
modelo2.fit(X2, y2)

# Realizamos las mismas transformaciones en el conjunto de testeo
y_test, X_test = (
    Formula(formula2)
    .get_model_matrix(df_test)
)

# Predicciones
y_pred = modelo2.predict(X_test)

# Bondad del ajuste
r2 = r2_score(y_test, y_pred)
print('R^2: ', r2)
ecm = mean_squared_error(y_test, y_pred)
print('Raiz cuadarada del ECM: ', np.sqrt(ecm))

## Ejercicio: MPG

Queremos comparar dos modelos lineales para predecir la eficiencia de motores.

1. Usando todas las variables numéricas: ["displacement", "horsepower", "weight", "acceleration", "model_year", "cylinders"]
2. Usando solo las variables:  ["weight", "acceleration", "model_year"]

Usando un esquema de entrenamiento, validación y testeo,
1. ¿Cuál modelo de menores errores en entrenamiento?
2. ¿Cuál modelo de menores errores en validación?
3. ¿Con cuál de los modelos nos quedamos?
4. ¿Qué error tiene nuestro modelo en testeo?


In [ ]:
mpg = sns.load_dataset("mpg").dropna()
mpg

In [ ]:
y = mpg["mpg"]
y

**Pregunta** Contestar utilizando un esquema adecuado: ¿es conveniente agregar la variable `origin` al modelo?